# OpenSpliceAI Delta Score Comparison: Limb vs Neuron Model

This notebook compares the delta scores from OpenSpliceAI predictions between Limb and Neuron tissue-specific models.

**Delta Scores:**
- DS_AG: Delta score (acceptor gain)
- DS_AL: Delta score (acceptor loss)
- DS_DG: Delta score (donor gain)
- DS_DL: Delta score (donor loss)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

# Set plot style
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 150

## 1. Define Helper Functions

In [ ]:
def parse_vcf(filepath):
    """Parse VCF file and extract relevant information"""
    records = []
    
    with open(filepath, 'r') as f:
        for line in f:
            if line.startswith('#'):
                continue
            
            fields = line.strip().split('\t')
            chrom, pos, var_id = fields[0], fields[1], fields[2]
            ref, alt = fields[3], fields[4]
            info = fields[7]
            
            # Parse INFO field
            info_dict = {}
            for item in info.split(';'):
                if '=' in item:
                    key, value = item.split('=', 1)
                    info_dict[key] = value
            
            # Extract CONSEQUENCE
            consequence = info_dict.get('CONSEQUENCE', '')
            
            # Extract PHENOTYPES
            phenotypes = info_dict.get('PHENOTYPES', '')
            has_limb = 'Abnormality_of_limbs' in phenotypes
            has_nervous = 'Abnormality_of_the_nervous_system' in phenotypes
            
            # Parse OpenSpliceAI - take the first prediction if multiple
            openspliceai = info_dict.get('OpenSpliceAI', '')
            if openspliceai:
                # Handle multiple predictions (separated by comma)
                first_pred = openspliceai.split(',')[0]
                parts = first_pred.split('|')
                if len(parts) >= 6:
                    try:
                        ds_ag = float(parts[2])  # Delta score acceptor gain
                        ds_al = float(parts[3])  # Delta score acceptor loss
                        ds_dg = float(parts[4])  # Delta score donor gain
                        ds_dl = float(parts[5])  # Delta score donor loss
                        
                        records.append({
                            'var_id': var_id,
                            'chrom': chrom,
                            'pos': pos,
                            'ref': ref,
                            'alt': alt,
                            'consequence': consequence,
                            'has_limb': has_limb,
                            'has_nervous': has_nervous,
                            'DS_AG': ds_ag,
                            'DS_AL': ds_al,
                            'DS_DG': ds_dg,
                            'DS_DL': ds_dl
                        })
                    except (ValueError, IndexError) as e:
                        print(f"Warning: Could not parse OpenSpliceAI for variant {var_id}: {e}")
    
    return pd.DataFrame(records)


def assign_phenotype_category(row):
    """Assign phenotype category based on limb and nervous system flags"""
    has_limb = row['has_limb']
    has_nervous = row['has_nervous']
    
    if has_limb and has_nervous:
        return 'Both'
    elif has_limb:
        return 'Limb only'
    elif has_nervous:
        return 'Nervous only'
    else:
        return 'Neither'

In [ ]:
def plot_comparison(df_limb, df_neuron, title_suffix='', ax=None, score_col='DS_AG'):
    """Plot comparison scatter plot for a specific delta score"""
    # Merge on var_id
    merged = pd.merge(df_limb, df_neuron, on='var_id', suffixes=('_limb', '_neuron'))
    
    # Assign phenotype category
    merged['category'] = merged.apply(
        lambda row: assign_phenotype_category({
            'has_limb': row['has_limb_limb'],
            'has_nervous': row['has_nervous_limb']
        }), axis=1
    )
    
    # Define colors
    colors = {
        'Neither': '#888888',
        'Limb only': '#e41a1c',
        'Nervous only': '#377eb8',
        'Both': '#4daf4a'
    }
    
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 8))
    
    # Plot each category
    for category in ['Neither', 'Limb only', 'Nervous only', 'Both']:
        mask = merged['category'] == category
        subset = merged[mask]
        ax.scatter(
            subset[f'{score_col}_limb'],
            subset[f'{score_col}_neuron'],
            c=colors[category],
            label=f'{category} (n={len(subset)})',
            alpha=0.6,
            s=30,
            edgecolors='none'
        )
    
    # Add diagonal line
    max_val = max(merged[f'{score_col}_limb'].max(), merged[f'{score_col}_neuron'].max())
    min_val = min(merged[f'{score_col}_limb'].min(), merged[f'{score_col}_neuron'].min())
    ax.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.5, linewidth=1)
    
    ax.set_xlabel(f'{score_col} (Limb model)', fontsize=11)
    ax.set_ylabel(f'{score_col} (Neuron model)', fontsize=11)
    ax.set_title(f'{score_col} Comparison{title_suffix}', fontsize=12)
    ax.legend(loc='upper left', fontsize=9)
    ax.set_aspect('equal', adjustable='box')
    
    # Calculate correlation
    corr = merged[f'{score_col}_limb'].corr(merged[f'{score_col}_neuron'])
    ax.text(0.95, 0.05, f'r = {corr:.4f}', transform=ax.transAxes, 
            fontsize=10, ha='right', va='bottom',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    return merged

## 2. Load and Parse VCF Files

In [ ]:
# Define file paths - modify these paths as needed
LIMB_VCF = '/mnt/user-data/uploads/limb_400_transfer.vcf'
NEURON_VCF = '/mnt/user-data/uploads/neuron_400_transfer.vcf'

# Or use your local paths:
# LIMB_VCF = '/home1/xyf/data/openspliceai_tissue_data/variant/limb_400_train.vcf'
# NEURON_VCF = '/home1/xyf/data/openspliceai_tissue_data/variant/neuron_400_train.vcf'

In [ ]:
# Parse VCF files
print("Parsing limb VCF file...")
df_limb = parse_vcf(LIMB_VCF)
print(f"Loaded {len(df_limb)} variants from limb model")

print("\nParsing neuron VCF file...")
df_neuron = parse_vcf(NEURON_VCF)
print(f"Loaded {len(df_neuron)} variants from neuron model")

## 3. Summary Statistics

In [ ]:
print("=== Limb model delta scores ===")
display(df_limb[['DS_AG', 'DS_AL', 'DS_DG', 'DS_DL']].describe())

print("\n=== Neuron model delta scores ===")
display(df_neuron[['DS_AG', 'DS_AL', 'DS_DG', 'DS_DL']].describe())

In [ ]:
# Phenotype distribution
df_limb['category'] = df_limb.apply(assign_phenotype_category, axis=1)

print("=== Phenotype Distribution ===")
print(df_limb['category'].value_counts())

In [ ]:
# Consequence distribution
print("=== Consequence Distribution ===")
print(df_limb['consequence'].value_counts())

## 4. All Variants Comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 14))

score_cols = ['DS_AG', 'DS_AL', 'DS_DG', 'DS_DL']
score_names = ['Acceptor Gain', 'Acceptor Loss', 'Donor Gain', 'Donor Loss']

for idx, (score_col, score_name) in enumerate(zip(score_cols, score_names)):
    ax = axes[idx // 2, idx % 2]
    plot_comparison(df_limb, df_neuron, title_suffix=f'\n(All variants, n={len(df_limb)})', 
                   ax=ax, score_col=score_col)

fig.suptitle('Limb vs Neuron Model Delta Score Comparison\n(All Variants)', 
              fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Splice Donor Variant Comparison

In [ ]:
# Filter for splice donor variants
df_limb_donor = df_limb[df_limb['consequence'] == 'splice_donor_variant_Likely_LOF']
df_neuron_donor = df_neuron[df_neuron['consequence'] == 'splice_donor_variant_Likely_LOF']

print(f"Splice donor variants: {len(df_limb_donor)}")

fig, axes = plt.subplots(2, 2, figsize=(14, 14))

for idx, (score_col, score_name) in enumerate(zip(score_cols, score_names)):
    ax = axes[idx // 2, idx % 2]
    plot_comparison(df_limb_donor, df_neuron_donor, 
                   title_suffix=f'\n(splice_donor_variant_Likely_LOF, n={len(df_limb_donor)})', 
                   ax=ax, score_col=score_col)

fig.suptitle('Limb vs Neuron Model Delta Score Comparison\n(splice_donor_variant_Likely_LOF)', 
              fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6. Splice Acceptor Variant Comparison

In [ ]:
# Filter for splice acceptor variants
df_limb_acceptor = df_limb[df_limb['consequence'] == 'splice_acceptor_variant_Likely_LOF']
df_neuron_acceptor = df_neuron[df_neuron['consequence'] == 'splice_acceptor_variant_Likely_LOF']

print(f"Splice acceptor variants: {len(df_limb_acceptor)}")

fig, axes = plt.subplots(2, 2, figsize=(14, 14))

for idx, (score_col, score_name) in enumerate(zip(score_cols, score_names)):
    ax = axes[idx // 2, idx % 2]
    plot_comparison(df_limb_acceptor, df_neuron_acceptor, 
                   title_suffix=f'\n(splice_acceptor_variant_Likely_LOF, n={len(df_limb_acceptor)})', 
                   ax=ax, score_col=score_col)

fig.suptitle('Limb vs Neuron Model Delta Score Comparison\n(splice_acceptor_variant_Likely_LOF)', 
              fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 7. Combined Comparison Grid (All, Splice Donor, Splice Acceptor)

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(20, 15))

colors = {'Neither': '#888888', 'Limb only': '#e41a1c', 
         'Nervous only': '#377eb8', 'Both': '#4daf4a'}

# Merge all data
merged_all = pd.merge(df_limb, df_neuron, on='var_id', suffixes=('_limb', '_neuron'))
merged_all['category'] = merged_all.apply(
    lambda row: assign_phenotype_category({
        'has_limb': row['has_limb_limb'],
        'has_nervous': row['has_nervous_limb']
    }), axis=1
)

merged_donor = pd.merge(df_limb_donor, df_neuron_donor, on='var_id', suffixes=('_limb', '_neuron'))
merged_donor['category'] = merged_donor.apply(
    lambda row: assign_phenotype_category({
        'has_limb': row['has_limb_limb'],
        'has_nervous': row['has_nervous_limb']
    }), axis=1
)

merged_acceptor = pd.merge(df_limb_acceptor, df_neuron_acceptor, on='var_id', suffixes=('_limb', '_neuron'))
merged_acceptor['category'] = merged_acceptor.apply(
    lambda row: assign_phenotype_category({
        'has_limb': row['has_limb_limb'],
        'has_nervous': row['has_nervous_limb']
    }), axis=1
)

datasets = [
    (merged_all, 'All'),
    (merged_donor, 'Donor'),
    (merged_acceptor, 'Acceptor')
]

row_labels = ['All Variants', 'Splice Donor LOF', 'Splice Acceptor LOF']

for row_idx, (merged, label) in enumerate(datasets):
    for col_idx, score_col in enumerate(score_cols):
        ax = axes[row_idx, col_idx]
        
        for category in ['Neither', 'Limb only', 'Nervous only', 'Both']:
            mask = merged['category'] == category
            subset = merged[mask]
            ax.scatter(subset[f'{score_col}_limb'], subset[f'{score_col}_neuron'],
                      c=colors[category], alpha=0.5, s=15, edgecolors='none',
                      label=f'{category}' if row_idx == 0 and col_idx == 0 else '')
        
        if len(merged) > 0:
            max_val = max(merged[f'{score_col}_limb'].max(), merged[f'{score_col}_neuron'].max())
            min_val = min(merged[f'{score_col}_limb'].min(), merged[f'{score_col}_neuron'].min())
            ax.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.5, linewidth=1)
            corr = merged[f'{score_col}_limb'].corr(merged[f'{score_col}_neuron'])
            ax.set_title(f'{score_col}\n({label}, r={corr:.4f})', fontsize=10)
        
        ax.set_xlabel('Limb', fontsize=9)
        if col_idx == 0:
            ax.set_ylabel(f'{row_labels[row_idx]}\nNeuron', fontsize=9)
        else:
            ax.set_ylabel('Neuron', fontsize=9)

# Add legend
handles = [plt.scatter([], [], c=colors[cat], s=50, label=cat) 
           for cat in ['Neither', 'Limb only', 'Nervous only', 'Both']]
fig.legend(handles=handles, labels=['Neither', 'Limb only', 'Nervous only', 'Both'],
           loc='upper center', ncol=4, bbox_to_anchor=(0.5, 0.02), fontsize=10)

fig.suptitle('OpenSpliceAI Delta Score Comparison: Limb vs Neuron Model\n', 
              fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0.05, 1, 0.98])
plt.show()

## 8. Score Differences Analysis

In [ ]:
print("=== Score Differences (Limb - Neuron) ===")

for score_col in score_cols:
    diff = merged_all[f'{score_col}_limb'] - merged_all[f'{score_col}_neuron']
    corr = merged_all[f'{score_col}_limb'].corr(merged_all[f'{score_col}_neuron'])
    print(f"\n{score_col}:")
    print(f"  Mean difference: {diff.mean():.6f}")
    print(f"  Std difference: {diff.std():.6f}")
    print(f"  Max positive diff: {diff.max():.4f}")
    print(f"  Max negative diff: {diff.min():.4f}")
    print(f"  Correlation: {corr:.4f}")

---

# Part 2: Phenotype-Specific Comparison (Limb only vs Nervous only)

## 9. Filter Data by Phenotype

In [ ]:
# Filter for Limb only and Nervous only
merged_limb_only = merged_all[merged_all['category'] == 'Limb only'].copy()
merged_nervous_only = merged_all[merged_all['category'] == 'Nervous only'].copy()

print(f"Limb only variants: {len(merged_limb_only)}")
print(f"Nervous only variants: {len(merged_nervous_only)}")

In [ ]:
# Detailed statistics
print("=== Limb only Delta Score Statistics ===")
for score_col in score_cols:
    diff = merged_limb_only[f'{score_col}_limb'] - merged_limb_only[f'{score_col}_neuron']
    corr = merged_limb_only[f'{score_col}_limb'].corr(merged_limb_only[f'{score_col}_neuron'])
    print(f"  {score_col}: mean_diff={diff.mean():.6f}, std={diff.std():.6f}, r={corr:.4f}")

print("\n=== Nervous only Delta Score Statistics ===")
for score_col in score_cols:
    diff = merged_nervous_only[f'{score_col}_limb'] - merged_nervous_only[f'{score_col}_neuron']
    corr = merged_nervous_only[f'{score_col}_limb'].corr(merged_nervous_only[f'{score_col}_neuron'])
    print(f"  {score_col}: mean_diff={diff.mean():.6f}, std={diff.std():.6f}, r={corr:.4f}")

## 10. Limb Only Phenotype Comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 12))

for idx, (score_col, score_name) in enumerate(zip(score_cols, score_names)):
    ax = axes[idx // 2, idx % 2]
    
    x = merged_limb_only[f'{score_col}_limb']
    y = merged_limb_only[f'{score_col}_neuron']
    
    ax.scatter(x, y, c='#e41a1c', alpha=0.6, s=40, edgecolors='white', linewidth=0.5)
    
    # Add diagonal line
    max_val = max(x.max(), y.max())
    min_val = min(x.min(), y.min())
    ax.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.5, linewidth=1)
    
    # Calculate stats
    corr = x.corr(y)
    mean_diff = (x - y).mean()
    
    ax.set_xlabel(f'{score_col} (Limb model)', fontsize=11)
    ax.set_ylabel(f'{score_col} (Neuron model)', fontsize=11)
    ax.set_title(f'{score_col} ({score_name})', fontsize=12)
    
    stats_text = f'r = {corr:.4f}\nMean diff = {mean_diff:.4f}\nn = {len(merged_limb_only)}'
    ax.text(0.95, 0.05, stats_text, transform=ax.transAxes, 
            fontsize=10, ha='right', va='bottom',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    ax.set_aspect('equal', adjustable='box')

fig.suptitle(f'Limb vs Neuron Model Delta Score Comparison\n(Limb only phenotype, n={len(merged_limb_only)})', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 11. Nervous Only Phenotype Comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 12))

for idx, (score_col, score_name) in enumerate(zip(score_cols, score_names)):
    ax = axes[idx // 2, idx % 2]
    
    x = merged_nervous_only[f'{score_col}_limb']
    y = merged_nervous_only[f'{score_col}_neuron']
    
    ax.scatter(x, y, c='#377eb8', alpha=0.6, s=40, edgecolors='white', linewidth=0.5)
    
    # Add diagonal line
    max_val = max(x.max(), y.max())
    min_val = min(x.min(), y.min())
    ax.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.5, linewidth=1)
    
    # Calculate stats
    corr = x.corr(y)
    mean_diff = (x - y).mean()
    
    ax.set_xlabel(f'{score_col} (Limb model)', fontsize=11)
    ax.set_ylabel(f'{score_col} (Neuron model)', fontsize=11)
    ax.set_title(f'{score_col} ({score_name})', fontsize=12)
    
    stats_text = f'r = {corr:.4f}\nMean diff = {mean_diff:.4f}\nn = {len(merged_nervous_only)}'
    ax.text(0.95, 0.05, stats_text, transform=ax.transAxes, 
            fontsize=10, ha='right', va='bottom',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    ax.set_aspect('equal', adjustable='box')

fig.suptitle(f'Limb vs Neuron Model Delta Score Comparison\n(Nervous only phenotype, n={len(merged_nervous_only)})', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 12. Side-by-Side Phenotype Comparison

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

colors = {'Limb only': '#e41a1c', 'Nervous only': '#377eb8'}

# Row 0: Limb only
for idx, (score_col, score_name) in enumerate(zip(score_cols, score_names)):
    ax = axes[0, idx]
    
    x = merged_limb_only[f'{score_col}_limb']
    y = merged_limb_only[f'{score_col}_neuron']
    
    ax.scatter(x, y, c=colors['Limb only'], alpha=0.6, s=40, 
              edgecolors='white', linewidth=0.5)
    
    max_val = max(x.max(), y.max()) if len(x) > 0 else 1
    min_val = min(x.min(), y.min()) if len(x) > 0 else 0
    ax.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.5, linewidth=1)
    
    corr = x.corr(y) if len(x) > 1 else 0
    mean_diff = (x - y).mean() if len(x) > 0 else 0
    
    ax.set_title(f'{score_col}\n(r={corr:.4f}, Δ={mean_diff:.4f})', fontsize=10)
    ax.set_xlabel('Limb model', fontsize=9)
    if idx == 0:
        ax.set_ylabel(f'Limb only (n={len(merged_limb_only)})\nNeuron model', fontsize=10)
    else:
        ax.set_ylabel('Neuron model', fontsize=9)

# Row 1: Nervous only
for idx, (score_col, score_name) in enumerate(zip(score_cols, score_names)):
    ax = axes[1, idx]
    
    x = merged_nervous_only[f'{score_col}_limb']
    y = merged_nervous_only[f'{score_col}_neuron']
    
    ax.scatter(x, y, c=colors['Nervous only'], alpha=0.6, s=40, 
              edgecolors='white', linewidth=0.5)
    
    max_val = max(x.max(), y.max()) if len(x) > 0 else 1
    min_val = min(x.min(), y.min()) if len(x) > 0 else 0
    ax.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.5, linewidth=1)
    
    corr = x.corr(y) if len(x) > 1 else 0
    mean_diff = (x - y).mean() if len(x) > 0 else 0
    
    ax.set_title(f'{score_col}\n(r={corr:.4f}, Δ={mean_diff:.4f})', fontsize=10)
    ax.set_xlabel('Limb model', fontsize=9)
    if idx == 0:
        ax.set_ylabel(f'Nervous only (n={len(merged_nervous_only)})\nNeuron model', fontsize=10)
    else:
        ax.set_ylabel('Neuron model', fontsize=9)

fig.suptitle('Phenotype-Specific Delta Score Comparison: Limb vs Neuron Model', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 13. Overlay Comparison (Limb only + Nervous only)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 14))

colors = {'Limb only': '#e41a1c', 'Nervous only': '#377eb8'}

for idx, (score_col, score_name) in enumerate(zip(score_cols, score_names)):
    ax = axes[idx // 2, idx % 2]
    
    # Plot Nervous only first (background)
    x_n = merged_nervous_only[f'{score_col}_limb']
    y_n = merged_nervous_only[f'{score_col}_neuron']
    ax.scatter(x_n, y_n, c=colors['Nervous only'], alpha=0.5, s=30, 
              edgecolors='none', label=f'Nervous only (n={len(merged_nervous_only)})')
    
    # Plot Limb only on top
    x_l = merged_limb_only[f'{score_col}_limb']
    y_l = merged_limb_only[f'{score_col}_neuron']
    ax.scatter(x_l, y_l, c=colors['Limb only'], alpha=0.7, s=40, 
              edgecolors='white', linewidth=0.5, label=f'Limb only (n={len(merged_limb_only)})')
    
    # Add diagonal line
    all_x = pd.concat([x_l, x_n])
    all_y = pd.concat([y_l, y_n])
    max_val = max(all_x.max(), all_y.max())
    min_val = min(all_x.min(), all_y.min())
    ax.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.5, linewidth=1)
    
    # Calculate stats for both
    corr_l = x_l.corr(y_l) if len(x_l) > 1 else 0
    corr_n = x_n.corr(y_n) if len(x_n) > 1 else 0
    diff_l = (x_l - y_l).mean() if len(x_l) > 0 else 0
    diff_n = (x_n - y_n).mean() if len(x_n) > 0 else 0
    
    ax.set_xlabel(f'{score_col} (Limb model)', fontsize=11)
    ax.set_ylabel(f'{score_col} (Neuron model)', fontsize=11)
    ax.set_title(f'{score_col} ({score_name})', fontsize=12)
    
    # Add stats box
    stats_text = (f'Limb only: r={corr_l:.4f}, Δ={diff_l:.4f}\n'
                 f'Nervous only: r={corr_n:.4f}, Δ={diff_n:.4f}')
    ax.text(0.95, 0.05, stats_text, transform=ax.transAxes, 
            fontsize=9, ha='right', va='bottom',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    ax.legend(loc='upper left', fontsize=9)
    ax.set_aspect('equal', adjustable='box')

fig.suptitle('Limb only vs Nervous only: Delta Score Comparison\n(Limb model vs Neuron model)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 14. Summary Table

In [ ]:
# Create summary table
summary_data = []

for score_col in score_cols:
    # Limb only
    diff_l = merged_limb_only[f'{score_col}_limb'] - merged_limb_only[f'{score_col}_neuron']
    corr_l = merged_limb_only[f'{score_col}_limb'].corr(merged_limb_only[f'{score_col}_neuron'])
    
    # Nervous only
    diff_n = merged_nervous_only[f'{score_col}_limb'] - merged_nervous_only[f'{score_col}_neuron']
    corr_n = merged_nervous_only[f'{score_col}_limb'].corr(merged_nervous_only[f'{score_col}_neuron'])
    
    # All
    diff_all = merged_all[f'{score_col}_limb'] - merged_all[f'{score_col}_neuron']
    corr_all = merged_all[f'{score_col}_limb'].corr(merged_all[f'{score_col}_neuron'])
    
    summary_data.append({
        'Score': score_col,
        'All_mean_diff': f'{diff_all.mean():.6f}',
        'All_corr': f'{corr_all:.4f}',
        'Limb_mean_diff': f'{diff_l.mean():.6f}',
        'Limb_corr': f'{corr_l:.4f}',
        'Nervous_mean_diff': f'{diff_n.mean():.6f}',
        'Nervous_corr': f'{corr_n:.4f}'
    })

summary_df = pd.DataFrame(summary_data)
print("=== Summary Comparison Table ===")
display(summary_df)